In [ ]:
import requests
import os
from pathlib import Path

data_dir = Path("data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

taxi_file = data_dir / "taxi_data.parquet"
zone_file = data_dir / "taxi_zone_lookup.csv"

# Only download taxi data if not already present
if not taxi_file.exists():
    url1 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
    response1 = requests.get(url1)
    response1.raise_for_status()
    taxi_file.write_bytes(response1.content)
    print("Taxi data downloaded.")
else:
    print("Taxi data already exists. Skipping download.")

# Only download zone lookup if not already present
if not zone_file.exists():
    url2 = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
    response2 = requests.get(url2)
    response2.raise_for_status()
    zone_file.write_bytes(response2.content)
    print("Zone lookup downloaded.")
else:
    print("Zone lookup already exists. Skipping download.")

print("Download check complete!")

## Part 1: Distributed Data Processing with Spark

Task 1.1

In [ ]:
import os
import sys
import subprocess

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PATH"] = (
    r"C:\Program Files\Java\jdk-17\bin;"
    r"C:\hadoop\bin;"
    + os.environ["PATH"]
)

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])
print("Python exe:", sys.executable)
print("winutils exists:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode())

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[1]") \
    .appName("TaxiAnalysis") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.sql.shuffle.partitions", "1") \
    .getOrCreate()

df = spark.read.parquet("data/raw/taxi_data.parquet")
df.printSchema()

In [ ]:
row_count = df.count()
partition_count = df.rdd.getNumPartitions()

print("Rows:", row_count)
print("Partitions:", partition_count)

In [ ]:
import time
import pandas as pd

# Pandas load time
start = time.time()
df = pd.read_parquet("data/raw/taxi_data.parquet")
pandas_time = time.time() - start

# Spark load time
start = time.time()
df = spark.read.parquet("data/raw/taxi_data.parquet")
df.count()
spark_time = time.time() - start

print(f"Spark load time: {spark_time:.2f} seconds")
print(f"Pandas load time: {pandas_time:.2f} seconds")

task 1.2

In [ ]:
from pyspark.sql import functions as F

initial_count = df.count()
print(f"Initial rows: {initial_count}")

# setting critical columns
critical_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "trip_distance",
    "tip_amount"
]

# drop nulls
df_clean = df.dropna(subset=critical_cols)
after_nulls = df_clean.count()
print(f"Rows after null removal: {after_nulls}")
print(f"Rows removed for nulls: {initial_count - after_nulls}")


# clean rows
df_clean = df_clean.filter(F.col("trip_distance") > 0)
after_distance = df_clean.count()
print(f"Rows removed for non-positive distance: {after_nulls - after_distance}")

df_clean = df_clean.filter(F.col("fare_amount") >= 0)
after_negative_fare = df_clean.count()
print(f"Rows removed for negative fare: {after_distance - after_negative_fare}")

df_clean = df_clean.filter(F.col("fare_amount") <= 500)
after_high_fare = df_clean.count()
print(f"Rows removed for fare > 500: {after_negative_fare - after_high_fare}")

df_clean = df_clean.filter(
    F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
after_bad_time = df_clean.count()
print(f"Rows removed for dropoff before pickup: {after_high_fare - after_bad_time}")

Derived Columns

In [ ]:
from pyspark.sql import functions as F

df_clean = df_clean.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("tpep_dropoff_datetime") -
        F.unix_timestamp("tpep_pickup_datetime")
    ) / 60.0)

df_clean = df_clean.withColumn(
    "trip_speed_mph",
    F.when(
        F.col("trip_duration_minutes") > 0,
        F.col("trip_distance") / (F.col("trip_duration_minutes") / 60.0)
    ).otherwise(0.0))

df_clean = df_clean.withColumn(
    "pickup_hour",
    F.hour("tpep_pickup_datetime"))

df_clean = df_clean.withColumn(
    "pickup_day_of_week",
    F.dayofweek("tpep_pickup_datetime"))

df_clean = df_clean.withColumn(
    "tip_percentage",
    F.when(
        F.col("fare_amount") > 1, 
        (F.col("tip_amount") / F.col("fare_amount")) * 100.0
    ).otherwise(0.0))

final_count = df_clean.count()
print(f"Final cleaned rows: {final_count}")

df_clean.select(
    "trip_duration_minutes",
    "trip_speed_mph",
    "pickup_hour",
    "pickup_day_of_week",
    "tip_percentage"
).show(5, truncate=False)

In [ ]:
df_clean.select(
    F.min("trip_duration_minutes"),
    F.min("trip_speed_mph"),
    F.min("tip_percentage"),
    F.max("tip_percentage")
).show()

task 1.3

In [ ]:
df_clean.createOrReplaceTempView("taxi_trips")

Query 1: What are the top 10 busiest pickup hours, and what is the average fare and tip 
percentage for each? 

In [ ]:
query1 = """
SELECT
    pickup_hour,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM taxi_trips
GROUP BY pickup_hour
ORDER BY trip_count DESC
LIMIT 10
"""

busiest_hours = spark.sql(query1)
busiest_hours.show(truncate=False)

Busiest hours are in the afternoon into the evening (from 15:00 to 19:00), being usual rush hour travel patterns. While the average fare is relatively consistent, the tip percentages seem to trend higher as it gets later into the pickup hours.

Query 2: Which day of the week has the highest average trip speed? Include average distance and duration. 

In [ ]:
query2 = """
SELECT
    pickup_day_of_week,
    ROUND(AVG(trip_speed_mph), 2) AS avg_speed_mph,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_minutes
FROM taxi_trips
GROUP BY pickup_day_of_week
ORDER BY avg_speed_mph DESC
"""

highest_avg_trip_speed = spark.sql(query2)
highest_avg_trip_speed.show(truncate=False)

Tuesdays seem to have a good flow of traffic leading to efficient travel, when compared to the similar trip duration of days like Wenesday and Thursday, that have a much lower avg speed and fairly shorter average distance but ultimately a very similar average trip duration.

Query 3: Using a window function, rank the top 5 pickup locations by total revenue for each day of the week.

In [ ]:
query3 = """
WITH revenue_by_day AS (
    SELECT
        pickup_day_of_week,
        PULocationID,
        ROUND(SUM(total_amount), 2) AS total_revenue
    FROM taxi_trips
    GROUP BY pickup_day_of_week, PULocationID
),
ranked_locations AS (
    SELECT
        pickup_day_of_week,
        PULocationID,
        total_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY pickup_day_of_week
            ORDER BY total_revenue DESC
        ) AS revenue_rank
    FROM revenue_by_day
)
SELECT
    pickup_day_of_week,
    PULocationID,
    total_revenue,
    revenue_rank
FROM ranked_locations
WHERE revenue_rank <= 5
ORDER BY pickup_day_of_week, revenue_rank
"""

top_5_PULocations = spark.sql(query3)
top_5_PULocations.show(50, truncate=False)

Pickup locations IDed 132 followed by 138 for each day of the week have the highest revenues. Location 132 by a large margin at that, with cases such as on Saturday, you can see that 132 has about 3 times the revenue as the next pickup Location below it.

Query 4: Calculate the cumulative trip count by hour of day (running total from hour 0 to 23). At what hour does the cumulative count surpass 50% of daily trips? 

In [ ]:
query4 = """
WITH hourly_counts AS (
    SELECT
        pickup_hour,
        COUNT(*) AS trip_count
    FROM taxi_trips
    GROUP BY pickup_hour
),
cumulative_counts AS (
    SELECT
        pickup_hour,
        trip_count,
        SUM(trip_count) OVER (
            ORDER BY pickup_hour
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_trip_count,
        SUM(trip_count) OVER () AS total_trip_count
    FROM hourly_counts
)
SELECT
    pickup_hour,
    trip_count,
    cumulative_trip_count,
    ROUND(cumulative_trip_count * 100.0 / total_trip_count, 2) AS cumulative_pct
FROM cumulative_counts
ORDER BY pickup_hour
"""

cumulative_trip_count = spark.sql(query4)
cumulative_trip_count.show(24, truncate=False)

In [ ]:
cumulative_trip_count.filter("cumulative_pct > 50").orderBy("pickup_hour").show(1, truncate=False)

It takes until about the start of afternoon rush hour to accumulate half of the day's trips. Which means that more than half the day has to pass before it gets to see half of it's demand for transportation.

Query 5: Compare average fare, distance, and tip percentage between short trips (<2 miles), medium trips (2–10 miles), and long trips (>10 miles). Which category has the highest tip percentage? 

In [ ]:
query5 = """
SELECT
    CASE
        WHEN trip_distance < 2 THEN 'Short (<2 miles)'
        WHEN trip_distance <= 10 THEN 'Medium (2-10 miles)'
        ELSE 'Long (>10 miles)'
    END AS trip_category,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM taxi_trips
GROUP BY
    CASE
        WHEN trip_distance < 2 THEN 'Short (<2 miles)'
        WHEN trip_distance <= 10 THEN 'Medium (2-10 miles)'
        ELSE 'Long (>10 miles)'
    END
ORDER BY avg_distance
"""

avg_distance_fare_tip = spark.sql(query5)
avg_distance_fare_tip.show(truncate=False)

The average distance for medium trips trends closer to the beginning of the threshold, showing that most people take shorter trips when combined with the amount of trips for short, but the average distance for long clears the beginning of the threshold number by a fairly large amount. This fact is not helped by the fact that this threshold, would be lumped with any large distance outliers.

Task 1.4

In [ ]:
import time

start = time.time()
spark.sql(query1).show()
no_cache_time = time.time() - start

print(f"No cache time: {no_cache_time:.2f} seconds")

In [ ]:
df_clean.cache()
df_clean.count()  

start = time.time()
spark.sql(query1).show()
cache_time = time.time() - start

print(f"With cache time: {cache_time:.2f} seconds")

In [ ]:
df_clean.write.mode("overwrite").partitionBy("pickup_hour").parquet("data/processed/taxi_partitioned")

df_partition = spark.read.parquet("data/processed/taxi_partitioned")

In [ ]:
df_partition.filter("pickup_hour = 17").show(5)
df_partition.filter("pickup_hour = 17").explain(True)

In [ ]:
spark.sql(query1).explain(True)

The physical plan includes a `FileScan` parquet to read the source data, a `Filter` to apply cleaning rules, and an `InMemoryTableScan` because the cleaned DataFrame was cached.  
Spark then uses `HashAggregate` for the grouped calculations and an Exchange operation for shuffling before returning the top 10 results.

## Part 2 RAG Pipeline over Transportation Documents

Task 2.1

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os

docs_path = "docs/"
documents = []
total_pages = 0

for file in os.listdir(docs_path):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(docs_path, file))
        docs = loader.load()
        documents.extend(docs)
        total_pages += len(docs)

full_text = " ".join([doc.page_content for doc in documents])

print("Total pages:", total_pages)
print("Total characters:", len(full_text))

Short section to see if any pages have issues

In [ ]:
quality_issues = {
    "empty_pages": 0,
    "very_short_pages": 0,
    "possible_scanned_pages": 0,
    "garbled_text_pages": 0
}

for doc in documents:
    text = doc.page_content.strip()

    # 1. Empty pages
    if len(text) == 0:
        quality_issues["empty_pages"] += 1

    # 2. Very short pages
    elif len(text) < 50:
        quality_issues["very_short_pages"] += 1

    # 3. Possible scanned pages
    if len(text) < 10:
        quality_issues["possible_scanned_pages"] += 1

    # 4. Garbled text detection (lots of weird characters)
    weird_chars = sum(1 for c in text if not c.isalnum() and c not in " .,!?-()")
    if len(text) > 0 and weird_chars / len(text) > 0.3:
        quality_issues["garbled_text_pages"] += 1

print("Quality Issues Report:")
for k, v in quality_issues.items():
    print(f"{k}: {v}")

Seemingly read all pages well enough

Task 2.2


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print("First chunk metadata:", chunks[0].metadata)
print("First chunk preview:\n", chunks[0].page_content[:300])

In [ ]:
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

print("Total chunks:", len(chunk_lengths))
print("Minimum chunk size:", min(chunk_lengths))
print("Maximum chunk size:", max(chunk_lengths))
print("Average chunk size:", sum(chunk_lengths) / len(chunk_lengths))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(chunk_lengths, bins=30, edgecolor="black")
plt.xlabel("Chunk size (characters)")
plt.ylabel("Frequency")
plt.title("Distribution of Chunk Sizes")
plt.show()

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_db",
    collection_name="transport_docs_1000"
)

print("Stored", len(chunks), "chunks in ChromaDB")

In [ ]:
print(chunks[0].metadata)

In [ ]:
import os

for chunk in chunks:
    if "source" in chunk.metadata:
        chunk.metadata["source"] = os.path.basename(chunk.metadata["source"])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import matplotlib.pyplot as plt
import os

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
splitter_500 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200
)

splitter_1000 = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splitter_2000 = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

chunks_500 = splitter_500.split_documents(documents)
chunks_1000 = splitter_1000.split_documents(documents)
chunks_2000 = splitter_2000.split_documents(documents)

for chunk_list in [chunks_500, chunks_1000, chunks_2000]:
    for chunk in chunk_list:
        if "source" in chunk.metadata:
            chunk.metadata["source"] = os.path.basename(chunk.metadata["source"])

print("Chunk counts:")
print("500  ->", len(chunks_500))
print("1000 ->", len(chunks_1000))
print("2000 ->", len(chunks_2000))

In [ ]:
chunk_lengths_1000 = [len(chunk.page_content) for chunk in chunks_1000]

print("Chunk size 1000 statistics")
print("Total chunks:", len(chunk_lengths_1000))
print("Minimum chunk size:", min(chunk_lengths_1000))
print("Maximum chunk size:", max(chunk_lengths_1000))
print("Average chunk size:", sum(chunk_lengths_1000) / len(chunk_lengths_1000))

plt.figure(figsize=(8, 5))
plt.hist(chunk_lengths_1000, bins=30, edgecolor="black")
plt.xlabel("Chunk size (characters)")
plt.ylabel("Frequency")
plt.title("Distribution of Chunk Sizes (chunk_size=1000)")
plt.show()

In [ ]:
vs_500 = Chroma.from_documents(
    documents=chunks_500,
    embedding=embedding_model,
    persist_directory="chroma_db",
    collection_name="transport_docs_500"
)

vs_1000 = Chroma.from_documents(
    documents=chunks_1000,
    embedding=embedding_model,
    persist_directory="chroma_db",
    collection_name="transport_docs_1000"
)

vs_2000 = Chroma.from_documents(
    documents=chunks_2000,
    embedding=embedding_model,
    persist_directory="chroma_db",
    collection_name="transport_docs_2000"
)

print("All vectorstores created.")

In [ ]:
query_1 = "How does the NYC Taxi and Limousine Commission regulate the number of for-hire vehicle licenses and what factors influence these decisions?"
query_2 = "What strategies and financial mechanisms are used to improve accessibility in NYC taxis, particularly for wheelchair accessible vehicles?"
query_3 = "How does tipping behaviour differ between traditional taxis and app-based ride services in NYC, and what factors explain these differences??"

queries = [query_1, query_2, query_3]

for i, query in enumerate(queries, start=1):
    print(f"Query {i}: {query}")

In [ ]:
print("RESULTS FOR CHUNK SIZE 500")

for idx, query in enumerate(queries, start=1):
    print("\n" + "=" * 80)
    print(f"Query {idx}: {query}")
    
    results = vs_500.similarity_search(query, k=3)
    
    for i, doc in enumerate(results, start=1):
        print(f"\nResult {i}")
        print("Source:", doc.metadata.get("source"))
        print("Page:", doc.metadata.get("page"))
        print(doc.page_content[:400].replace("\n", " "))

In [ ]:
print("RESULTS FOR CHUNK SIZE 1000")

for idx, query in enumerate(queries, start=1):
    print("\n" + "=" * 80)
    print(f"Query {idx}: {query}")
    
    results = vs_1000.similarity_search(query, k=3)
    
    for i, doc in enumerate(results, start=1):
        print(f"\nResult {i}")
        print("Source:", doc.metadata.get("source"))
        print("Page:", doc.metadata.get("page"))
        print(doc.page_content[:400].replace("\n", " "))

In [ ]:
print("RESULTS FOR CHUNK SIZE 2000")

for idx, query in enumerate(queries, start=1):
    print("\n" + "=" * 80)
    print(f"Query {idx}: {query}")
    
    results = vs_2000.similarity_search(query, k=3)
    
    for i, doc in enumerate(results, start=1):
        print(f"\nResult {i}")
        print("Source:", doc.metadata.get("source"))
        print("Page:", doc.metadata.get("page"))
        print(doc.page_content[:400].replace("\n", " "))

Three chunk sizes (500, 1000, and 2000) were evaluated using three policy-related queries. The 500-character chunks retrieved highly specific but often repetitive fragments, frequently returning duplicate sections and lacking sufficient context to fully answer the queries. The 2000-character chunks provided broader context but sometimes retrieved less relevant information, including large sections of unrelated text such as tables or general descriptions. The 1000-character configuration offered the best balance between relevance and context, consistently retrieving meaningful and interpretable sections without excessive noise. Therefore, a chunk size of 1000 was selected as the most effective for the RAG pipeline.

Task 2.3

In [ ]:
RAG_PROMPT = """
You are a helpful assistant answering questions about NYC transportation policy.

Use ONLY the context provided below to answer the question.
If the answer is not contained in the context, say: "I could not find the answer in the provided documents."

When answering:
1. Be concise but complete.
2. Do not use outside knowledge.
3. Cite the source filename(s) and page number(s) used.

Question:
{question}

Context:
{context}

Answer:
"""

def retrieve_chunks(query, vectorstore, k=4):
    results = vectorstore.similarity_search(query, k=k)
    return results

def format_context(docs):
    context_parts = []
    
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "Unknown source")
        page = doc.metadata.get("page", "Unknown page")
        text = doc.page_content.strip()
        
        context_parts.append(
            f"[Source {i}: {source}, page {page}]\n{text}"
        )
    
    return "\n\n".join(context_parts)

def extract_sources(docs):
    seen = set()
    sources = []
    
    for doc in docs:
        source = doc.metadata.get("source", "Unknown source")
        page = doc.metadata.get("page", "Unknown page")
        key = (source, page)
        
        if key not in seen:
            seen.add(key)
            sources.append({"source": source, "page": page})
    
    return sources

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = os.getenv("OPENAI_BASE_URL")
MODEL_NAME = os.getenv("OPENAI_MODEL")

In [ ]:
from openai import OpenAI
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)


def call_llm(prompt):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You answer only from provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=400
    )
    
    return response.choices[0].message.content

In [ ]:
def deduplicate_docs(docs):
    seen = set()
    unique_docs = []
    
    for doc in docs:
        key = (
            doc.metadata.get("source"),
            doc.metadata.get("page"),
            doc.page_content[:200]
        )
        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)
    
    return unique_docs

def rag_query(question, vectorstore, k=4):
    retrieved_docs = retrieve_chunks(question, vectorstore, k=k)
    retrieved_docs = deduplicate_docs(retrieved_docs)
    
    context = format_context(retrieved_docs)
    sources = extract_sources(retrieved_docs)
    
    prompt = RAG_PROMPT.format(
        question=question,
        context=context
    )
    
    answer = call_llm(prompt)
    
    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_docs": retrieved_docs,
        "prompt": prompt
    }

In [ ]:
def print_rag_result(result):
    print("=" * 100)
    print("QUESTION:")
    print(result["question"])
    
    print("\nGENERATED ANSWER:")
    print(result["answer"])
    
    print("\nSOURCES USED:")
    for src in result["sources"]:
        print(f"- {src['source']} (page {src['page']})")
    
    print("\nRETRIEVED CONTEXT CHUNKS:")
    for i, doc in enumerate(result["retrieved_docs"], start=1):
        print(f"\nChunk {i}")
        print("Source:", doc.metadata.get("source"))
        print("Page:", doc.metadata.get("page"))
        print(doc.page_content[:700].replace("\n", " "))

test_questions = [
    "How does the TLC regulate the number of for-hire vehicle licenses and what factors affect these decisions?",
    "What strategies are used to improve accessibility for wheelchair accessible vehicles in NYC taxis?",
    "How does tipping behaviour differ between traditional taxis and app-based ride services in NYC?",
    "What does the TLC annual report say about electric vehicle adoption in the for-hire industry?",
    "What safety issues and enforcement actions are described in the commuter van safety study?"
]

In [ ]:
rag_results = []

for question in test_questions:
    result = rag_query(question, vs_1000, k=4)
    rag_results.append(result)
    print_rag_result(result)

Task 2.4

In [ ]:
test_set = [
    {
        "question": "How does the TLC regulate the number of for-hire vehicle licenses and what factors influence these decisions?",
        "expected_answer": "The TLC regulates FHV licenses through periodic reviews and considers factors such as congestion, driver pay, license attrition, outer-borough service, EV demand, and charging infrastructure.",
        "expected_source": "license-pause-report-2024-02.pdf"
    },
    {
        "question": "Why was the issuance of new for-hire vehicle licenses paused, and what exceptions were allowed?",
        "expected_answer": "The issuance of new FHV licenses was paused to address congestion and related impacts, with exceptions made for wheelchair accessible vehicles and later for electric vehicles.",
        "expected_source": "license-pause-report-2024-02.pdf"
    },
    {
        "question": "What trends in trip volume and vehicle supply are described in the FHV license review?",
        "expected_answer": "Trip volumes increased post-pandemic, approaching pre-pandemic levels, while vehicle supply grew significantly due to new licenses, particularly EV-only licenses.",
        "expected_source": "license-pause-report-2024-02.pdf"
    },
    {
        "question": "How does tipping behaviour differ between traditional taxis and app-based ride services in NYC?",
        "expected_answer": "Traditional taxis show more predictable tipping behaviour due to in-vehicle payment prompts, while app-based services have more random and less predictable tipping because tipping is decoupled from the ride.",
        "expected_source": "ONE SIZE FITS NONE MODELING NYC TAXI TIPS.pdf"
    },
    {
        "question": "Why does the tipping study argue that a single universal tipping model is ineffective?",
        "expected_answer": "The study argues that a universal model is ineffective because different taxi categories exhibit distinct tipping behaviours, and combining them leads to misleading results due to Simpson’s paradox.",
        "expected_source": "ONE SIZE FITS NONE MODELING NYC TAXI TIPS.pdf"
    },
    {
        "question": "What financial mechanism is used to fund taxi accessibility programs in NYC?",
        "expected_answer": "Taxi accessibility programs are funded through a surcharge on yellow and green taxi trips, which contributes to the Taxi Improvement Fund and Street Hail Livery Improvement Fund.",
        "expected_source": "tif_report_2024.pdf"
    },
    {
        "question": "What incentives are provided to drivers and owners of wheelchair accessible vehicles?",
        "expected_answer": "Drivers receive per-trip payments for operating accessible vehicles, while owners receive financial incentives including upfront payments and ongoing bonuses for maintaining accessible vehicles in service.",
        "expected_source": "tif_report_2024.pdf"
    },
    {
        "question": "What does the TLC annual report say about electric vehicle adoption in the for-hire vehicle industry?",
        "expected_answer": "The report highlights rapid growth in EV adoption, driven by the Green Rides Initiative, with increasing EV trip share and expanded charging infrastructure.",
        "expected_source": "annual_report_2024.pdf"
    },
    {
        "question": "What is the Flex Fare program and how does it benefit drivers?",
        "expected_answer": "The Flex Fare program allows upfront pricing for taxi trips, similar to ride-hailing services, and increases driver earnings by improving pricing transparency and demand.",
        "expected_source": "annual_report_2024.pdf"
    },
    {
        "question": "What safety issues and enforcement actions are described in the commuter van safety study?",
        "expected_answer": "The study reports safety violations, minimal collisions, and enforcement actions such as targeted operations, towing, LIDAR enforcement, and joint operations with law enforcement agencies.",
        "expected_source": "commuter_van_safety_study_2024.pdf"
    }
]

In [ ]:
evaluation_results = []

for item in test_set:
    result = rag_query(item["question"], vs_1000, k=4)
    
    retrieved_sources = [doc.metadata.get("source") for doc in result["retrieved_docs"]]
    
    evaluation_results.append({
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "expected_source": item["expected_source"],
        "generated_answer": result["answer"],
        "retrieved_sources": retrieved_sources,
        "retrieved_docs": result["retrieved_docs"]
    })

In [ ]:
for i, item in enumerate(evaluation_results, start=1):
    print("=" * 100)
    print(f"Test Case {i}")
    print("Question:", item["question"])
    print("\nExpected Source:", item["expected_source"])
    print("Retrieved Sources:", item["retrieved_sources"])
    print("\nExpected Answer:")
    print(item["expected_answer"])
    print("\nGenerated Answer:")
    print(item["generated_answer"])
    print()

In [ ]:
manual_eval = [
    {
        "question": test_set[0]["question"],
        "retrieval_correct": True,
        "answer_correct": True,
        "failure_type": "None"
    },
    {
        "question": test_set[1]["question"],
        "retrieval_correct": True,
        "answer_correct": False,
        "failure_type": "Generation Failure"
    },
    {
        "question": test_set[2]["question"],
        "retrieval_correct": True,
        "answer_correct": True,
        "failure_type": "None"
    },
    {
        "question": test_set[3]["question"],
        "retrieval_correct": True,
        "answer_correct": True,
        "failure_type": "None"
    },
    {
        "question": test_set[4]["question"],
        "retrieval_correct": True,
        "answer_correct": True,
        "failure_type": "None"
    },
    {
        "question": test_set[5]["question"],
        "retrieval_correct": True,
        "answer_correct": False,
        "failure_type": "Generation Failure"
    },
    {
        "question": test_set[6]["question"],
        "retrieval_correct": False,
        "answer_correct": False,
        "failure_type": "Retrieval Failure"
    },
    {
        "question": test_set[7]["question"],
        "retrieval_correct": True,
        "answer_correct": True,
        "failure_type": "None"
    },
    {
        "question": test_set[8]["question"],
        "retrieval_correct": True,
        "answer_correct": True,
        "failure_type": "None"
    },
    {
        "question": test_set[9]["question"],
        "retrieval_correct": True,
        "answer_correct": False,
        "failure_type": "Generation Failure"
    }
]

In [ ]:
import pandas as pd

eval_df = pd.DataFrame(manual_eval)
eval_df

In [ ]:
retrieval_accuracy = eval_df["retrieval_correct"].mean() * 100
answer_accuracy = eval_df["answer_correct"].mean() * 100

overall_accuracy = (
    ((eval_df["retrieval_correct"]) & (eval_df["answer_correct"])).mean() * 100
)

print(f"Retrieval Accuracy: {retrieval_accuracy:.1f}%")
print(f"Answer Accuracy: {answer_accuracy:.1f}%")
print(f"Overall Accuracy: {overall_accuracy:.1f}%")

In [ ]:
print(eval_df["failure_type"].value_counts())

## Part 3 Integrated Analytics Application

Task 3.1

In [ ]:
import json

ROUTER_SYSTEM_PROMPT = """
You are a query router for an analytics system with two backends:

1. DATA:
   Use this when the question can be answered using the structured NYC taxi trip dataset.
   Examples:
   - average fare
   - trip counts
   - busiest pickup hour
   - average tip percentage
   - trip distance, duration, speed
   - trends by day/hour/location from the taxi data

2. DOCUMENT:
   Use this when the question can be answered using the transportation policy PDF documents only.
   Examples:
   - TLC regulations
   - accessibility policies
   - for-hire vehicle licensing rules
   - annual report findings
   - policy recommendations
   - funding mechanisms described in documents

3. HYBRID:
   Use this when the question requires BOTH the structured taxi dataset and the document corpus,
   or when the question is ambiguous and could reasonably require both.
   Examples:
   - compare actual taxi behaviour to policy recommendations
   - compare observed tipping patterns to what the documents say
   - combine operational data with regulatory context

Return ONLY valid JSON in this exact format:
{
  "category": "DATA" | "DOCUMENT" | "HYBRID",
  "reasoning": "short explanation"
}
"""

def call_router_llm(user_query):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": ROUTER_SYSTEM_PROMPT},
            {"role": "user", "content": f"Classify this query:\n{user_query}"}
        ],
        temperature=0,
        max_tokens=150
    )
    return response.choices[0].message.content.strip()

def parse_router_output(raw_output, original_query):
    try:
        parsed = json.loads(raw_output)
        category = parsed.get("category", "").strip().upper()
        reasoning = parsed.get("reasoning", "").strip()

        if category not in {"DATA", "DOCUMENT", "HYBRID"}:
            return {
                "query": original_query,
                "category": "HYBRID",
                "reasoning": "Invalid category returned by model, so defaulted to HYBRID.",
                "raw_output": raw_output
            }

        return {
            "query": original_query,
            "category": category,
            "reasoning": reasoning if reasoning else "No reasoning provided.",
            "raw_output": raw_output
        }

    except Exception:
        return {
            "query": original_query,
            "category": "HYBRID",
            "reasoning": "Model output could not be parsed as JSON, so defaulted to HYBRID.",
            "raw_output": raw_output
        }

def route_query(query):
    raw_output = call_router_llm(query)
    return parse_router_output(raw_output, query)

In [ ]:
sample_queries = [
    "What was the average fare on Mondays?",
    "What incentives are offered for wheelchair accessible vehicles?",
    "How do actual tipping patterns compare to TLC recommendations?"
]

for q in sample_queries:
    result = route_query(q)
    print("=" * 80)
    print("Query:", result["query"])
    print("Category:", result["category"])
    print("Reasoning:", result["reasoning"])
    print("Raw output:", result["raw_output"])

In [ ]:
router_test_set = [
    # DATA
    {"query": "What was the average fare on Mondays?", "expected": "DATA"},
    {"query": "Which pickup hour had the highest trip count?", "expected": "DATA"},
    {"query": "What is the average tip percentage for short trips?", "expected": "DATA"},
    {"query": "Which day of the week has the highest average trip speed?", "expected": "DATA"},
    {"query": "What are the top pickup locations by total revenue?", "expected": "DATA"},

    # DOCUMENT
    {"query": "What financial mechanism funds taxi accessibility programmes in NYC?", "expected": "DOCUMENT"},
    {"query": "What incentives are provided to owners of wheelchair accessible vehicles?", "expected": "DOCUMENT"},
    {"query": "How does the TLC regulate the number of for-hire vehicle licences?", "expected": "DOCUMENT"},
    {"query": "What does the TLC annual report say about electric vehicle adoption?", "expected": "DOCUMENT"},
    {"query": "What safety concerns are discussed in the commuter van safety study?", "expected": "DOCUMENT"},

    # HYBRID
    {"query": "How do actual tipping patterns compare to TLC recommendations?", "expected": "HYBRID"},
    {"query": "Do observed trip patterns support the policy focus on accessibility?", "expected": "HYBRID"},
    {"query": "How does taxi demand in the data compare with the licensing concerns described in the documents?", "expected": "HYBRID"},
    {"query": "Are the busiest travel periods consistent with the transportation issues highlighted in the reports?", "expected": "HYBRID"},
    {"query": "How do real fare and tip trends compare with what the policy documents say about driver incentives?", "expected": "HYBRID"},
]

In [ ]:
router_results = []

for item in router_test_set:
    prediction = route_query(item["query"])
    router_results.append({
        "query": item["query"],
        "expected": item["expected"],
        "predicted": prediction["category"],
        "correct": prediction["category"] == item["expected"],
        "reasoning": prediction["reasoning"]
    })

In [ ]:
import pandas as pd

router_results_df = pd.DataFrame(router_results)
router_results_df

In [ ]:
router_accuracy = router_results_df["correct"].mean() * 100
print(f"Router classification accuracy: {router_accuracy:.2f}%")

In [ ]:
router_results_df["predicted"].value_counts()

Task 3.2

In [ ]:
import re
import pandas as pd

DATA_SQL_SYSTEM_PROMPT = """
You translate natural language questions into valid Spark SQL.

Database/view available:
- taxi_trips

Important columns in taxi_trips:
- tpep_pickup_datetime
- tpep_dropoff_datetime
- PULocationID
- DOLocationID
- passenger_count
- trip_distance
- fare_amount
- tip_amount
- total_amount
- trip_duration_minutes
- trip_speed_mph
- pickup_hour
- pickup_day_of_week
- tip_percentage

Rules:
1. Return ONLY SQL.
2. Use Spark SQL syntax.
3. Query only from taxi_trips.
4. Do not invent columns.
5. Prefer readable aliases without double quotes.
6. If the question asks for highest, busiest, most common, top, or largest by frequency, use COUNT(*), GROUP BY, ORDER BY ... DESC, and LIMIT.
7. If the question asks for an average by category, use GROUP BY appropriately.
8. Make sure the SQL answers the exact analytical question, not a simplified version.
9. When returning pickup_day_of_week in results, convert it to a weekday name using: 1=Sunday, 2=Monday, 3=Tuesday, 4=Wednesday, 5=Thursday, 6=Friday, 7=Saturday.

Examples:

Question: Which pickup hour had the highest trip count?
SQL:
SELECT
    pickup_hour,
    COUNT(*) AS trip_count
FROM taxi_trips
GROUP BY pickup_hour
ORDER BY trip_count DESC
LIMIT 1;

Question: What was the average fare on Mondays?
SQL:
SELECT AVG(fare_amount) AS avg_fare
FROM taxi_trips
WHERE pickup_day_of_week = 2;
"""

ANSWER_SYSTEM_PROMPT = """
You are a data analyst.

Given a user's question and a SQL query result, write a short natural language answer.

Rules:
1. Base the answer only on the provided result.
2. Do not guess missing meanings.
3. If a column contains numeric day codes, only interpret them if a mapping is explicitly shown.
4. Preserve numeric values carefully and do not misread scientific notation.
5. If there are multiple rows, summarise the key findings clearly.
"""

In [ ]:
def call_llm_messages(messages, temperature=0, max_tokens=400):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content.strip()

def extract_sql(raw_text):
    text = raw_text.strip()
    
    if text.startswith("```"):
        text = text.replace("```sql", "").replace("```", "").strip()
    
    match = re.search(r"(SELECT|WITH)\b.*", text, flags=re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(0).strip()
    
    return text

def generate_sql_from_question(question, error_message=None):
    user_prompt = f"""
Question: {question}
Generate a valid Spark SQL query for this question.
Return only SQL.
"""
    
    if error_message:
        user_prompt += f"\nThe previous SQL failed with this error:\n{error_message}\nPlease correct it."

    raw_sql = call_llm_messages([
        {"role": "system", "content": DATA_SQL_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ], temperature=0, max_tokens=300)

    return extract_sql(raw_sql)

def execute_sql_with_retry(question):
    sql_query = generate_sql_from_question(question)
    
    try:
        result_df = spark.sql(sql_query)
        return {
            "success": True,
            "sql": sql_query,
            "result_df": result_df,
            "error": None,
            "retried": False
        }
    except Exception as e:
        first_error = str(e)
        
        retry_sql = generate_sql_from_question(question, error_message=first_error)
        
        try:
            result_df = spark.sql(retry_sql)
            return {
                "success": True,
                "sql": retry_sql,
                "result_df": result_df,
                "error": first_error,
                "retried": True
            }
        except Exception as e2:
            return {
                "success": False,
                "sql": retry_sql,
                "result_df": None,
                "error": f"First error: {first_error}\nRetry error: {str(e2)}",
                "retried": True
            }

def dataframe_to_text(result_df, max_rows=10):
    pdf = result_df.limit(max_rows).toPandas()
    if pdf.empty:
        return "No rows returned."
    return pdf.to_string(index=False)

def synthesise_data_answer(question, sql_query, result_text):
    prompt = f"""
User question:
{question}

Executed SQL:
{sql_query}

SQL result:
{result_text}

Write a short natural language answer based only on the SQL result.
"""
    
    return call_llm_messages([
        {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ], temperature=0, max_tokens=250)

def handle_data_query(question):
    execution = execute_sql_with_retry(question)
    
    if not execution["success"]:
        return {
            "question": question,
            "sql": execution["sql"],
            "raw_result": None,
            "answer": "The system could not generate a valid SQL query for this question.",
            "success": False,
            "error": execution["error"],
            "retried": execution["retried"]
        }
    
    raw_result = dataframe_to_text(execution["result_df"])
    answer = synthesise_data_answer(question, execution["sql"], raw_result)
    
    return {
        "question": question,
        "sql": execution["sql"],
        "raw_result": raw_result,
        "answer": answer,
        "success": True,
        "error": execution["error"],
        "retried": execution["retried"]
    }

In [ ]:
def print_data_query_result(result):
    print("=" * 100)
    print("QUESTION:")
    print(result["question"])
    
    print("\nGENERATED SQL:")
    print(result["sql"])
    
    if result["error"]:
        print("\nERROR / RETRY INFO:")
        print(result["error"])
        print(f"Retried: {result['retried']}")
    
    print("\nRAW RESULT:")
    print(result["raw_result"])
    
    print("\nFINAL ANSWER:")
    print(result["answer"])

In [ ]:
data_questions = [
    "What was the average fare on Mondays?",
    "Which pickup hour had the highest trip count?",
    "What is the average tip percentage for short trips under 2 miles?",
    "Which day of the week has the highest average trip speed?",
    "What are the top 5 pickup locations by total revenue?"
]

In [ ]:
data_query_results = []

for question in data_questions:
    result = handle_data_query(question)
    data_query_results.append(result)
    print_data_query_result(result)

In [ ]:
summary_rows = []

for r in data_query_results:
    summary_rows.append({
        "question": r["question"],
        "success": r["success"],
        "retried": r["retried"],
        "answer_preview": r["answer"][:120]
    })

pd.DataFrame(summary_rows)

In [ ]:
def handle_document_query(query):
    return rag_query(query, vs_1000, k=4)


def handle_hybrid_query(query):
    data_subquery_prompt = f"""
Rewrite the following HYBRID question into a DATA-ONLY analytical question
that can be answered from the taxi_trips Spark view alone.

Rules:
- Focus only on measurable trip behaviour in the dataset
- Do not mention documents, policies, TLC recommendations, or regulations
- Avoid calculations that directly divide tip_amount by fare_amount
- Prefer using already derived safe columns like tip_percentage, pickup_hour,
  pickup_day_of_week, trip_distance, trip_speed_mph, total_amount
- Return only the rewritten question

Original question:
{query}
"""

    data_subquery = call_llm_messages([
        {"role": "system", "content": "You rewrite mixed questions into safe dataset-only analytical questions."},
        {"role": "user", "content": data_subquery_prompt}
    ], temperature=0, max_tokens=120)

    data_result = handle_data_query(data_subquery)
    data_answer = data_result["answer"] if data_result["success"] else "No data answer available."

    doc_result = handle_document_query(query)
    doc_answer = doc_result["answer"]

    combined_prompt = f"""
User question:
{query}

DATA-ONLY SUBQUESTION:
{data_subquery}

DATA-BASED ANSWER:
{data_answer}

DOCUMENT-BASED ANSWER:
{doc_answer}

Write one final answer that combines both sources clearly.
Mention where the dataset evidence and document evidence agree or differ.
Base the answer only on the information above.
"""

    final_answer = call_llm_messages([
        {"role": "system", "content": "You combine data analysis and document-based insights into one clear answer."},
        {"role": "user", "content": combined_prompt}
    ], temperature=0, max_tokens=300)

    return {
        "data_subquery": data_subquery,
        "data_result": data_result,
        "doc_result": doc_result,
        "data_answer": data_answer,
        "doc_answer": doc_answer,
        "final_answer": final_answer
    }

def full_pipeline(query):
    classification = route_query(query)

    category = classification["category"]

    if category == "DATA":
        result = handle_data_query(query)
        final_answer = result["answer"]

    elif category == "DOCUMENT":
        result = handle_document_query(query)
        final_answer = result["answer"]

    else:  # HYBRID
        result = handle_hybrid_query(query)
        final_answer = result["final_answer"]

    return {
        "query": query,
        "category": category,
        "reasoning": classification["reasoning"],
        "result": result,
        "final_answer": final_answer
    }

In [ ]:
def print_full_pipeline_output(output):
    print("=" * 100)
    print("QUERY:")
    print(output["query"])

    print("\nCLASSIFICATION:")
    print(output["category"])
    print("Reasoning:", output["reasoning"])

    print("\nROUTING DECISION:")
    print(f"Routed to: {output['category']} pipeline")

    print("\nPIPELINE OUTPUT:")

    if output["category"] == "DATA":
        print("\nGENERATED SQL:")
        print(output["result"]["sql"])

        print("\nRAW RESULT:")
        print(output["result"]["raw_result"])

    elif output["category"] == "DOCUMENT":
        print("\nGENERATED ANSWER:")
        print(output["result"]["answer"])

        print("\nSOURCES USED:")
        for src in output["result"]["sources"]:
            print(f"- {src['source']} (page {src['page']})")

    else:
        print("\nDATA SUBQUESTION:")
        print(output["result"]["data_subquery"])

        print("\nDATA ANSWER:")
        print(output["result"]["data_answer"])

        print("\nDOCUMENT ANSWER:")
        print(output["result"]["doc_answer"])

        print("\nDOCUMENT SOURCES:")
        for src in output["result"]["doc_result"]["sources"]:
            print(f"- {src['source']} (page {src['page']})")

    print("\nFINAL ANSWER:")
    print(output["final_answer"])

In [ ]:
demo_queries = [
    # DATA
    "What was the average fare on Mondays?",
    "Which pickup hour has the highest trip count?",

    # DOCUMENT
    "What financial mechanism funds taxi accessibility programs in NYC?",
    "What incentives are provided for wheelchair accessible vehicles?",

    # HYBRID
    "How do actual tipping patterns compare to TLC recommendations?",
    "How does taxi demand in the data compare with issues discussed in the TLC reports?"
]

In [ ]:
demo_results = []

for q in demo_queries:
    result = full_pipeline(q)
    demo_results.append(result)
    print_full_pipeline_output(result)

The integrated system successfully combines structured analytics from Spark SQL with document-based retrieval using the RAG pipeline, providing a unified natural language interface for different types of queries. The query router was able to correctly classify queries into `DATA`, `DOCUMENT`, and `HYBRID` categories, enabling appropriate routing to the relevant backend.  
  
`DATA` queries were handled most effectively, with accurate SQL generation and meaningful analytical results.   
`DOCUMENT` queries were also processed through the RAG pipeline, though in some cases the system retrieved relevant sources but was unable to extract a precise answer, highlighting limitations in retrieval quality.
`HYBRID` queries demonstrated the ability to combine insights from both structured data and document sources. However, they were also the most challenging, as they required decomposing the original query into a data-focused and document-focused component. In some cases, the generated data sub-queries only partially aligned with the original intent, resulting in answers that did not fully capture the comparison being asked. Additionally, the system’s performance depends heavily on both the quality of retrieved document chunks and the correctness of LLM-generated SQL.

With more time, improvements could include enhancing retrieval accuracy through better chunking or re-ranking strategies, refining prompt engineering for query decomposition, and improving alignment between `HYBRID` query components. Additional validation of generated SQL and stronger handling of edge cases would further improve robustness.